# Changing tree depth (`max_depth`)

This presentation notebook runs the reusable scikit-learn experiment, then displays its exported tables and figures. Model selection uses stratified cross-validation on the training split; the held-out test set is not searched.

> Educational classification demo; not a medical diagnosis.

In [ ]:
from pathlib import Path
import subprocess
import sys

import pandas as pd
from IPython.display import Image, display

current = Path.cwd().resolve()
REPOSITORY_ROOT = current if (current / 'experiments').is_dir() else current.parent
CONFIG = REPOSITORY_ROOT / 'experiments/configs/max_depth.json'
RESULTS_DIR = REPOSITORY_ROOT / 'experiments/results/max_depth'
REPOSITORY_ROOT

## Run the reproducible experiment

In [ ]:
completed = subprocess.run(
    [sys.executable, str(REPOSITORY_ROOT / 'scripts/run_max_depth_experiment.py'), '--config', str(CONFIG)],
    cwd=REPOSITORY_ROOT,
    check=True,
    capture_output=True,
    text=True,
)
print(completed.stdout)

## Cross-validation results

The finite depth with the highest validation mean is selected. A shallower tree wins an exact tie.

In [ ]:
cv_results = pd.read_csv(RESULTS_DIR / 'cv_results.csv')
display(cv_results.style.format({
    'train_accuracy_mean': '{:.4f}',
    'train_accuracy_std': '{:.4f}',
    'validation_accuracy_mean': '{:.4f}',
    'validation_accuracy_std': '{:.4f}',
    'validation_error_rate': '{:.4f}',
}))

In [ ]:
display(Image(filename=str(RESULTS_DIR / 'accuracy_by_depth.png')))

## Final held-out comparison

Only the predeclared unlimited baseline and the CV-selected finite-depth model are evaluated here.

In [ ]:
final_comparison = pd.read_csv(RESULTS_DIR / 'final_comparison.csv')
display(final_comparison.style.format({
    'train_accuracy': '{:.4f}',
    'test_accuracy': '{:.4f}',
    'test_error_rate': '{:.4f}',
    'malignant_precision': '{:.4f}',
    'malignant_recall': '{:.4f}',
    'malignant_f1': '{:.4f}',
}))

In [ ]:
baseline = final_comparison.iloc[0]
selected = final_comparison.iloc[1]
print(f"Test accuracy change: {selected.test_accuracy - baseline.test_accuracy:+.4f}")
print(f"Test error-rate change: {selected.test_error_rate - baseline.test_error_rate:+.4f}")
print(f"Malignant recall change: {selected.malignant_recall - baseline.malignant_recall:+.4f}")
print(f"Leaf reduction: {int(baseline.n_leaves - selected.n_leaves)}")

## Selected tree

In [ ]:
display(Image(filename=str(RESULTS_DIR / 'selected_tree.png')))

## Interpretation checklist

- Small depths may underfit because both training and validation accuracy remain low.
- A widening train-validation gap at larger depths is evidence of overfitting.
- Report whether accuracy, malignant recall, and false negatives improve or worsen.
- Prefer the simpler tree when performance is tied, and do not claim clinical readiness.
- The current 80/20 split, seed 42, and accuracy selection rule remain provisional until decision D-006 is accepted.